# MongoDB Silver Validation

## Objective

This notebook validates the Silver dataset generated from the MongoDB source.

The purpose of this notebook is to verify that all approved transformation rules have been applied correctly.

No transformations are performed here.

This notebook only validates:

- Schema integrity
- Record count consistency
- Cleaning rules
- Derived columns
- Business flags
- Data quality
- Dataset readiness for Gold

## Note

A complete summary of this notebook can be found in the final section:
**Validation Summary**.

In [10]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("postgres_to_parquet_silver") \
    .master("spark://spark-master:7077") \
    .config("spark.default.parallelism", "4") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.driver.memory", "1g") \
    .config("spark.sql.autoBroadcastJoinThreshold", "10m") \
    .config("spark.sql.parquet.enableVectorizedReader", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minio") \
    .config("spark.hadoop.fs.s3a.secret.key", "minio123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

In [11]:
STORAGE_PROTOCOL = "s3a://"
BUCKET_NAME = "end-to-end-streaming-data-platform-bronze"
SOURCE_SUSTEM = "mongo"
FOLDER_NAME = "ingestion_data"
execution_date = "2026-07-13-Jul"
TABLE_NAME = "videos"

execution_date = "2026-07-13-Jul"
full_file_path = f"{STORAGE_PROTOCOL}{BUCKET_NAME}/{SOURCE_SUSTEM}/{FOLDER_NAME}={execution_date}/{TABLE_NAME}.parquet"

#Pv = "s3a://end-to-end-streaming-data-platform-bronze/mongo/ingestion_data=2026-07-13-Jul/videos.parquet"

In [12]:
dfv = spark.read.parquet(full_file_path).cache()
dfv.createOrReplaceTempView("v_table")

In [13]:
# ============================================================
# MongoDB → Silver Transform
# ============================================================

silver_df ="""

WITH base AS (

    SELECT

        -- Identifiers
        video_id,
        user_id,
        match_id,

        -- Content
        content_type,
        status,

        CASE
            WHEN title IS NULL OR TRIM(title) = '' THEN NULL
            ELSE title
        END AS title,

        description,

        CASE
            WHEN duration_sec IS NULL THEN NULL
            WHEN duration_sec BETWEEN 1 AND 10800 THEN duration_sec
            ELSE NULL
        END AS duration_sec,

        created_at,
        last_updated,

        -- Competition
        competition.id    AS competition_id,
        competition.name  AS competition_name,
        competition.teams AS competition_teams,

        -- Technical
        technical.max_resolution AS technical_max_resolution,
        technical.codec          AS technical_codec,

        -- Stats
        stats.views AS stats_views,
        stats.likes AS stats_likes,

        -- Arrays
        languages,
        ARRAY_DISTINCT(tags) AS tags

    FROM v_table

)

SELECT

    video_id,
    user_id,
    match_id,

    content_type,
    status,
    title,
    description,
    duration_sec,
    created_at,
    CAST(created_at AS DATE) AS created_date,
    last_updated,
    CAST(last_updated AS DATE) AS last_updated_date,

    competition_id,
    competition_name,
    competition_teams,

    technical_max_resolution,
    technical_codec,

    stats_views,
    stats_likes,

    languages,
    tags,

    CASE
        WHEN stats_views IS NOT NULL
         AND stats_likes > stats_views
        THEN TRUE
        ELSE FALSE
    END AS flag_invalid_views_likes,

    CASE
        WHEN content_type = 'live'
         AND match_id IS NULL
        THEN TRUE
        ELSE FALSE
    END AS flag_missing_live_match,

    CASE
        WHEN (content_type = 'live'      AND status <> 'streaming')
          OR (content_type = 'replay'    AND status <> 'archived')
          OR (content_type = 'highlight' AND status <> 'ended')
        THEN TRUE
        ELSE FALSE
    END AS flag_invalid_content_status,

    CASE
    WHEN competition_id = 'RSL'
         AND (
             array_contains(tags, 'Kings_Cup')
             OR array_contains(tags, 'Super_Cup')
         )
    THEN TRUE

    WHEN competition_id = 'Kings_Cup'
         AND (
             array_contains(tags, 'RSL')
             OR array_contains(tags, 'Super_Cup')
         )
    THEN TRUE

    WHEN competition_id = 'Super_Cup'
         AND (
             array_contains(tags, 'RSL')
             OR array_contains(tags, 'Kings_Cup')
         )
    THEN TRUE

    ELSE FALSE
    END AS flag_invalid_competition_tags,
    

    current_timestamp() AS silver_processed_at

FROM base
"""

In [14]:
video_df = spark.sql(silver_df)

In [16]:
# Dataset Overview

print("=" * 60)
print("Mongo Silver Dataset Overview")
print("=" * 60)

print(f"Rows    : {video_df.count()}")
print(f"Columns : {len(video_df.columns)}")

video_df.printSchema()

Mongo Silver Dataset Overview


[Stage 1:=============================>                             (1 + 1) / 2]

Rows    : 50000
Columns : 26
root
 |-- video_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- match_id: string (nullable = true)
 |-- content_type: string (nullable = true)
 |-- status: string (nullable = true)
 |-- title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- duration_sec: integer (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- created_date: date (nullable = true)
 |-- last_updated: timestamp (nullable = true)
 |-- last_updated_date: date (nullable = true)
 |-- competition_id: string (nullable = true)
 |-- competition_name: string (nullable = true)
 |-- competition_teams: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- technical_max_resolution: string (nullable = true)
 |-- technical_codec: string (nullable = true)
 |-- stats_views: integer (nullable = true)
 |-- stats_likes: integer (nullable = true)
 |-- languages: array (nullable = true)
 |    |-- element: string (containsNul

In [18]:
# Schema Validation

expected_columns = {
    "video_id",
    "user_id",
    "match_id",

    "content_type",
    "status",
    "title",
    "description",
    "duration_sec",

    "created_at",
    "created_date",

    "last_updated",
    "last_updated_date",

    "competition_id",
    "competition_name",
    "competition_teams",

    "technical_max_resolution",
    "technical_codec",

    "stats_views",
    "stats_likes",

    "languages",
    "tags",

    "flag_invalid_views_likes",
    "flag_missing_live_match",
    "flag_invalid_content_status",
    "flag_invalid_competition_tags",
    "silver_processed_at"
}

actual_columns = set(video_df.columns)

missing_columns = expected_columns - actual_columns
extra_columns = actual_columns - expected_columns

print("=" * 60)
print("Schema Validation")
print("=" * 60)

print(f"Expected Columns : {len(expected_columns)}")
print(f"Actual Columns   : {len(actual_columns)}")

if not missing_columns:
    print("No Missing Columns")
else:
    print("Missing Columns:")
    for col in sorted(missing_columns):
        print(f"   - {col}")

if not extra_columns:
    print("No Unexpected Columns")
else:
    print("Extra Columns:")
    for col in sorted(extra_columns):
        print(f"   - {col}")

Schema Validation
Expected Columns : 26
Actual Columns   : 26
No Missing Columns
No Unexpected Columns


In [19]:
# Nested Structure Validation
print("=" * 60)
print("Nested Structure Validation")
print("=" * 60)

nested_columns = [
    "competition",
    "technical",
    "stats"
]

existing_nested = [
    col
    for col in nested_columns
    if col in video_df.columns
]

if len(existing_nested) == 0:
    print("All nested structs were flattened successfully.")
else:
    print("Nested structs still exist:")
    for col in existing_nested:
        print(f"   - {col}")

Nested Structure Validation
All nested structs were flattened successfully.


In [20]:
print("=" * 60)
print("Row Count Validation")
print("=" * 60)

bronze_count = dfv.count()
silver_count = video_df.count()

print(f"Bronze Rows : {bronze_count}")
print(f"Silver Rows : {silver_count}")
print(f"Difference  : {bronze_count - silver_count}")

if bronze_count == silver_count:
    print("No records were removed.")
else:
    print("Row count mismatch.")

Row Count Validation
Bronze Rows : 50000
Silver Rows : 50000
Difference  : 0
No records were removed.


In [33]:
# ============================================================
# Null Validation
# ============================================================

print("=" * 70)
print("Null Validation")
print("=" * 70)

total_rows = video_df.count()

null_summary = []

for column in video_df.columns:

    null_count = video_df.filter(F.col(column).isNull()).count()

    null_percentage = round((null_count / total_rows) * 100, 2)

    null_summary.append(
        (
            column,
            null_count,
            f"{null_percentage}%"
        )
    )

null_df = spark.createDataFrame(
    null_summary,
    ["Column", "Null Count", "Null %"]
)


null_df.orderBy(
        F.desc("Null Count")
).show(truncate=False)

Null Validation


+-----------------------------+----------+------+
|Column                       |Null Count|Null %|
+-----------------------------+----------+------+
|duration_sec                 |18244     |36.49%|
|description                  |3971      |7.94% |
|stats_views                  |2593      |5.19% |
|match_id                     |2538      |5.08% |
|title                        |1483      |2.97% |
|technical_max_resolution     |1036      |2.07% |
|stats_likes                  |0         |0.0%  |
|created_at                   |0         |0.0%  |
|languages                    |0         |0.0%  |
|competition_id               |0         |0.0%  |
|tags                         |0         |0.0%  |
|video_id                     |0         |0.0%  |
|flag_invalid_views_likes     |0         |0.0%  |
|competition_name             |0         |0.0%  |
|flag_missing_live_match      |0         |0.0%  |
|created_date                 |0         |0.0%  |
|flag_invalid_content_status  |0         |0.0%  |


In [35]:
# ============================================================
# Numeric & Data Type Validation
# ============================================================

print("=" * 70)
print("Data Type Validation")
print("=" * 70)

expected_types = {
    "created_at": "timestamp",
    "last_updated": "timestamp",

    "created_date": "date",
    "last_updated_date": "date",

    "duration_sec": "int",

    "stats_views": "int",
    "stats_likes": "int"
}

schema = dict(video_df.dtypes)

validation = []

for column, expected in expected_types.items():

    actual = schema.get(column)

    validation.append(
        (
            column,
            expected,
            actual,
            expected == actual
        )
    )

validation_df = spark.createDataFrame(
    validation,
    [
        "Column",
        "Expected",
        "Actual",
        "PASS"
    ]
)

validation_df.show(truncate=False)

Data Type Validation
+-----------------+---------+---------+----+
|Column           |Expected |Actual   |PASS|
+-----------------+---------+---------+----+
|created_at       |timestamp|timestamp|true|
|last_updated     |timestamp|timestamp|true|
|created_date     |date     |date     |true|
|last_updated_date|date     |date     |true|
|duration_sec     |int      |int      |true|
|stats_views      |int      |int      |true|
|stats_likes      |int      |int      |true|
+-----------------+---------+---------+----+



In [26]:
# ============================================================
# Cleaning Validation
# ============================================================

print("=" * 70)
print("Cleaning Validation")
print("=" * 70)

checks = [

    (
        "Empty Titles",
        video_df.filter(
            F.col("title") == ""
        ).count()
    ),

    (
        "Negative Duration",
        video_df.filter(
            F.col("duration_sec") <= 0
        ).count()
    ),

    (
        "Duration > 10800",
        video_df.filter(
            F.col("duration_sec") > 10800
        ).count()
    )

]

for name, value in checks:

    if value == 0:
        print(f"✅ {name}: PASS")
    else:
        print(f"❌ {name}: {value}")

Cleaning Validation
✅ Empty Titles: PASS
✅ Negative Duration: PASS
✅ Duration > 10800: PASS


In [36]:
# ============================================================
# Duplicate Validation
# ============================================================

print("=" * 70)
print("Duplicate Validation")
print("=" * 70)

duplicate_video_ids = (

    video_df

    .groupBy("video_id")

    .count()

    .filter(
        F.col("count") > 1
    )

)

duplicates = duplicate_video_ids.count()

if duplicates == 0:

    print("video_id is unique.")

else:

    print(f"Duplicate video_id values: {duplicates}")

duplicate_video_ids.show(truncate=False)

Duplicate Validation
video_id is unique.
+--------+-----+
|video_id|count|
+--------+-----+
+--------+-----+



In [37]:
# ============================================================
# Business Flags Validation
# ============================================================

print("=" * 70)
print("Business Flags Validation")
print("=" * 70)

total_rows = video_df.count()

business_flags = [

    "flag_invalid_views_likes",

    "flag_missing_live_match",

    "flag_invalid_content_status",

    "flag_invalid_competition_tags"

]

summary = []

for flag in business_flags:

    flagged = video_df.filter(F.col(flag)).count()

    percentage = round(
        flagged / total_rows * 100,
        2
    )

    summary.append(

        (
            flag,
            flagged,
            percentage
        )

    )

summary_df = spark.createDataFrame(

    summary,

    [
        "Business Flag",
        "Affected Records",
        "Percentage"
    ]

)

summary_df.show(truncate=False)

Business Flags Validation
+-----------------------------+----------------+----------+
|Business Flag                |Affected Records|Percentage|
+-----------------------------+----------------+----------+
|flag_invalid_views_likes     |2406            |4.81      |
|flag_missing_live_match      |834             |1.67      |
|flag_invalid_content_status  |33299           |66.6      |
|flag_invalid_competition_tags|28821           |57.64     |
+-----------------------------+----------------+----------+



In [41]:
print("=" * 70)
print("Invalid Content Type / Status Combinations")
print("=" * 70)

video_df.filter(
    (
        (F.col("content_type") == "live") &
        (F.col("status") != "streaming")
    )
    |
    (
        (F.col("content_type") == "replay") &
        (F.col("status") != "archived")
    )
    |
    (
        (F.col("content_type") == "highlight") &
        (F.col("status") != "ended")
    )
).groupBy(
    "content_type",
    "status"
).count().orderBy(
    F.desc("count")
).show(truncate=False)

Invalid Content Type / Status Combinations
+------------+---------+-----+
|content_type|status   |count|
+------------+---------+-----+
|replay      |ended    |5684 |
|live        |ended    |5563 |
|highlight   |archived |5550 |
|highlight   |streaming|5542 |
|replay      |streaming|5487 |
|live        |archived |5473 |
+------------+---------+-----+



In [40]:
# ============================================================
# Flag Validation Summary
# ============================================================

flag_summary = [
    ("flag_invalid_views_likes",
     video_df.filter(F.col("flag_invalid_views_likes")).count()),

    ("flag_missing_live_match",
     video_df.filter(F.col("flag_missing_live_match")).count()),

    ("flag_invalid_content_status",
     video_df.filter(F.col("flag_invalid_content_status")).count()),

    ("flag_invalid_competition_tags",
     video_df.filter(F.col("flag_invalid_competition_tags")).count())
]

flag_summary_df = spark.createDataFrame(
    flag_summary,
    ["Flag", "Invalid Records"]
)

flag_summary_df = flag_summary_df.withColumn(
    "Status",
    F.when(F.col("Invalid Records") == 0, "PASS")
     .otherwise("REVIEW")
)

print("=" * 70)
print("Flag Validation Summary")
print("=" * 70)

flag_summary_df.show(truncate=False)

Flag Validation Summary
+-----------------------------+---------------+------+
|Flag                         |Invalid Records|Status|
+-----------------------------+---------------+------+
|flag_invalid_views_likes     |2406           |REVIEW|
|flag_missing_live_match      |834            |REVIEW|
|flag_invalid_content_status  |33299          |REVIEW|
|flag_invalid_competition_tags|28821          |REVIEW|
+-----------------------------+---------------+------+



In [30]:
# ============================================================
# Derived Columns Validation
# ============================================================

print("=" * 70)
print("Derived Columns Validation")
print("=" * 70)

created_errors = video_df.filter(

    F.to_date("created_at")

    !=

    F.col("created_date")

).count()

updated_errors = video_df.filter(

    F.to_date("last_updated")

    !=

    F.col("last_updated_date")

).count()

print(

    f"created_date mismatches : {created_errors}"

)

print(

    f"last_updated_date mismatches : {updated_errors}"

)

Derived Columns Validation
created_date mismatches : 0
last_updated_date mismatches : 0


In [31]:
# ============================================================
# Dataset Health
# ============================================================

print("=" * 70)
print("Dataset Health")
print("=" * 70)

total_flagged = video_df.filter(

    F.col("flag_invalid_views_likes")

    |

    F.col("flag_missing_live_match")

    |

    F.col("flag_invalid_content_status")

    |

    F.col("flag_invalid_competition_tags")

).count()

print(

    f"Total Records : {total_rows}"

)

print(

    f"Flagged Records : {total_flagged}"

)

print(

    f"Healthy Records : {total_rows - total_flagged}"

)

print(

    f"Healthy % : {round((total_rows-total_flagged)/total_rows*100,2)}%"

)

Dataset Health
Total Records : 50000
Flagged Records : 43407
Healthy Records : 6593
Healthy % : 13.19%


In [32]:
# ============================================================
# Final Validation Report
# ============================================================

print("=" * 70)
print("MongoDB Silver Validation Report")
print("=" * 70)

print("Schema Validation            ✅ PASS")

print("Flatten Validation          ✅ PASS")

print("Record Count Validation     ✅ PASS")

print("Data Type Validation        ✅ PASS")

print("Cleaning Validation         ✅ PASS")

print("Duplicate Validation        ✅ PASS")

print("Derived Columns Validation  ✅ PASS")

print("Business Flags             ⚠️ REVIEW REQUIRED")

print()

print("Dataset Status")

print("READY FOR GOLD")

MongoDB Silver Validation Report
Schema Validation            ✅ PASS
Flatten Validation          ✅ PASS
Record Count Validation     ✅ PASS
Data Type Validation        ✅ PASS
Cleaning Validation         ✅ PASS
Duplicate Validation        ✅ PASS
Derived Columns Validation  ✅ PASS
Business Flags             ⚠️ REVIEW REQUIRED

Dataset Status
READY FOR GOLD


In [42]:
video_df.write \
    .mode("overwrite") \
    .parquet("s3a://end-to-end-streaming-data-platform-silver/mongo/videos_silver.parquet")